# Übung 03: Ein KI-System für eine medizinische Fragestellung


In dieser Übung wollen wir ein einfaches komplettes KI-System aufbauen. Dazu untersuchen wir exemplarisch, ob sich ein binärer Herzbefund aus bereits vorliegenden Untersuchungsdaten vorhersagen lässt.


## Aufgabe 1: Klausuraufgabe

Ein Forschungsteam möchte nach einer Belastungsuntersuchung vorhersagen, ob ein koronarer Herzbefund vorliegt. Die Untersuchungsergebnisse liegen in Tabellenform vor. Eine Zeile beschreibt eine Person. Die Zielklasse `herzkrankheit` ist **0** (kein entsprechender Befund) oder **1** (Befund vorhanden). Der Datensatz enthält 303 Personen, davon 164 mit Klasse 0 und 139 mit Klasse 1.

Jede Spalte repräsentiert ein erfasstes Feature. Die folgenden Spalten stehen zur Verfügung.

| Spalte | Bedeutung |
|:--|:--|
| `age` | Alter in Jahren |
| `sex` | Im Ausgangsdatensatz codiertes Geschlecht: 0 = weiblich, 1 = männlich |
| `cp` | Brustschmerztyp: 1 = typische Angina, 2 = atypische Angina, 3 = nicht-anginöser Schmerz, 4 = asymptomatisch |
| `trestbps` | Ruheblutdruck bei Aufnahme in mmHg |
| `chol` | Serumcholesterin in mg/dl |
| `thalach` | maximale Herzfrequenz während des Belastungstests |
| `exang` | belastungsinduzierte Angina: 0 = nein, 1 = ja |
| `oldpeak` | belastungsbedingte ST-Streckensenkung gegenüber Ruhe |
| `Krankenhaus` | aus welchem Krankenhaus stammen die Daten: Sacred Heart, Sankt-Helena, County General oder Seattle Grace |
| `urspruenglicher_befund` | ursprünglicher Diagnosewert 0–4, aus dem `herzkrankheit` gebildet wurde |

Das Modell soll für **neue Personen nach Abschluss des Belastungstests** eingesetzt werden.


### 1a) Zielgröße und Merkmale (5 Punkte)

1. Erläutern Sie, ob es sich um eine Regressions- oder Klassifikationsaufgabe handelt.
2. Nennen Sie zwei Merkmale, die für die Fragestellung plausibel relevant sein könnten, und begründen Sie Ihre Auswahl.
3. Nennen Sie zwei Merkmale, die bei der Modellbildung für Probleme sorgen könnten, und begründen Sie Ihre Auswahl.


Ihre Lösung:


### 1b) Datenbegutachtung und Vorverarbeitung (4 Punkte)

1. Ordnen Sie jedes Feature einer der Kategorien Numerisch, Kategorial, oder Nominal zu.
2. Ordnen Sie vier verschiedene Features je einer medizinischen Datenart zu.


Ihre Lösung:


### 1c) Einen Klassifikator bauen (6 Punkte)

Ein Decision-Tree-Klassifikator soll aus den zehn Merkmalen den binären Herzbefund vorhersagen. Sie können davon ausgehen, dass alle Merkmale vollständig erfasst wurden und keine Werte fehlen. Die folgende Tabelle nennt verschiedene Arbeitsschritte, um den Decision-Tree-Klassifikator aus den Daten zu entwickeln. Streichen Sie die ungeeigneten Schritte und nummerieren Sie die übrigen in einer sinnvollen Reihenfolge. Gibt es Schritte, die mehrmals ausgeführt werden sollten?

| Nummer | Schritt |
|:--:|:--|
|  | A: Den Decision Tree auf allen verfügbaren Daten trainieren. |
|  | B: Den Decision Tree auf allen verfügbaren Daten evaluieren. |
|  | C: Die Personen in Trainings-, Validierungs- und Testdaten aufteilen. |
|  | D: Alle weiblichen Personen aus dem Datensatz löschen. |
|  | E: Den Decision Tree auf dem Validierungsdatensatz evaluieren und die Parameter des Decision Tree basierend auf diesem Ergebnis anpassen. |
|  | F: Die Trainingsgüte als unabhängige Schätzung der späteren Leistung verwenden. |
|  | G: Den Decision Tree auf dem Trainingsdatensatz trainieren.|
|  | H: Die Features `Krankenhaus` und `urspruenglicher_befund` löschen. |
|  | I: Features normalisieren. |
|  | J: Die Features `trestbps` und `exang` löschen. |


Ihre Lösung:


## Aufgabe 2: Vorrechenaufgabe - eine einfache Baseline

In dieser Aufgabe liegt der in Aufgabe 1 eingeführte Beispieldatensatz tatsächlich vor, in Form der bereitgestellten CSV-Datei `herzkrankheit_cleveland.csv`. Eine Zeile beschreibt eine Person. Die elf Spalten enthalten zehn Features sowie `herzkrankheit`, die binäre Zielklasse. Für die Baseline übernehmen wir zunächst alle Eingabespalten ohne fachliche Merkmalsprüfung.

Die Daten sind eine abgewandelte Version der [Cleveland-Teilmenge des UCI-Heart-Disease-Datensatzes](https://archive.ics.uci.edu/dataset/45/heart+disease) (CC BY 4.0).


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix, accuracy_score, recall_score

# Wenn ihr eure Daten in Google Drive hochgeladen habt (empfohlen - eventuell Ordner noch anpassen; MyDrive heißt in der deutschen Drive-Version "Meine Ablage"):
from google.colab import drive
drive.mount("/content/drive")
dateipfad = "/content/drive/MyDrive/herzkrankheit_cleveland.csv"
daten = pd.read_csv(dateipfad)
display(daten.head())
print("Form:", daten.shape)
print("Zielklassen:", daten["herzkrankheit"].value_counts().sort_index().to_dict())
print("Fehlende Werte:", int(daten.isna().sum().sum()))


Um die Features von der Zielgröße zu trennen, teilen wir das DataFrame in die Featurematrix `X` und den Zielvektor `y`. Zunächst reservieren wir 20 % der Personen als Testdaten. Vom Rest verwenden wir ein Viertel zur Validierung. Dafür nutzen wir die Funktion `train_test_split` der Bibiothek `sklearn`. Der Parameter `stratify` hält die Klassenanteile in den Teilmengen ungefähr konstant. Der `random_state` macht den Vergleich reproduzierbar. Die Testdaten bleiben zunächst unangetastet.


In [ ]:
X = daten.drop(columns="herzkrankheit")
y = daten["herzkrankheit"]

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.25, stratify=y_trainval, random_state=42
)
print("Training, Validierung, Test:", len(X_train), len(X_val), len(X_test))


Ein kleiner Entscheidungsbaum ist eine leicht nachvollziehbare Referenz. Er kann jedoch textbasierte Features nicht unmittelbar verarbeiten. Deshalb codiert die vorbereitete Pipeline die vier Krankenhausnamen als getrennte 0/1-Spalten. Erneut nutzen wir die Bibliothek `sklearn`, insbesondere die Klasse `DecisionTreeClassifier`, sowie den `ColumnTranformer` und einen `OneHotEncoder` für die Vorverarbeitung; alles organisiert in einer `Pipeline`.
Der Parameter `max_depth=3` begrenzt die Baumtiefe. `fit` lernt ausschließlich aus den Trainingsdaten; die Vorhersagen prüfen wir auf der Validierung.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

# Technische Codierung der Textspalte; alle übrigen Eingaben werden übernommen.
baseline = Pipeline([
    ("vorverarbeitung", ColumnTransformer(
        [("krankenhaus", OneHotEncoder(handle_unknown="ignore"), ["Krankenhaus"])],
        remainder="passthrough"
    )),
    ("baum", DecisionTreeClassifier(max_depth=3, random_state=42)),
])
baseline.fit(X_train, y_train)
vorhersage_basis_val = baseline.predict(X_val)

matrix_basis_val = confusion_matrix(y_val, vorhersage_basis_val, labels=[0, 1])
display(pd.DataFrame(matrix_basis_val, index=["wahr 0", "wahr 1"], columns=["Vorhersage 0", "Vorhersage 1"]))
ConfusionMatrixDisplay(matrix_basis_val, display_labels=["kein Befund", "Befund"]).plot(cmap="Blues", values_format="d")
plt.title("Baseline auf den Validierungsdaten")
plt.show()
print("Accuracy:", round(accuracy_score(y_val, vorhersage_basis_val), 3))
print("Recall für Befund:", round(recall_score(y_val, vorhersage_basis_val), 3))


In der Matrix stehen links oben richtig negative und rechts unten richtig positive Fälle. Links unten sind falsch negative Fälle und rechts oben falsch positive. Die Accuracy erfasst alle korrekten Entscheidungen; der Recall der Klasse 1 betrachtet nur tatsächlich positive Fälle.

Die Validierungsmatrix enthält keinerlei falsch klassifizierte Samples. Die Accuracy beträgt 100 %, ebenso der Recall. Entweder, wir haben das Problem perfekt gelöst oder ein paar entscheidende Fehler gemacht.


## Aufgabe 3: Eigenarbeit – ein verwendbares Modell prüfen

Die Baseline berücksichtigt weder die Sinnhaftigkeit noch die Datenherkunft oder -verfügbarkeit. Eine hohe Kennzahl allein belegt deshalb noch kein gutes KI-System. Prüft zuerst die Merkmale (dabei könnt ihr euch an den Ergebnissen der Klausuraufgabe orientieren) und vergleicht danach einen bereinigten Baum mit einer logistischen Regression. Ziel ist ein Vergleich anhand der Validierungsdaten. Entscheidet vor der abschließenden Testauswertung, welches der beiden bereinigten Modelle ihr bevorzugt.


### 3a) Daten und Merkmale prüfen

Untersucht die Trainingsdaten mithilfe des vorbereiteten Codes. Die erste Tabelle zeigt eine generelle Datenübersicht mithilfe der Funktion `describe` der Bibliothek `pandas`, inklusive Klassenhäufigkeit, Gruppenhäufigkeit, Wertebereiche und statistischen Kennzahlen der Spalten.

Tabelle zwei zeigt die Häufigkeit der Werte für die Spalte `urspruenglicher_befund`, getrennt nach den Werten für `herzkrankheit`.

Tabelle drei zeigt die Häufigkeit der Werte für die Spalte `sex`, getrennt nach den Werten für `herzkrankheit`.


In [ ]:
trainingsdaten = pd.concat([X_train, y_train], axis=1)
display(trainingsdaten.describe(include="all").T)
display(pd.crosstab(X_train["urspruenglicher_befund"], y_train,
                    rownames=["urspruenglicher_befund"], colnames=["herzkrankheit"]))
display(pd.crosstab(X_train["sex"], y_train, rownames=["sex"], colnames=["herzkrankheit"]))


Wählt Merkmale aus, die nach dieser Prüfung vor einer sinnvollen Modellbildung entfernt werden sollten. Tragt die übrigen Merkmale in die Liste `merkmale` im vorbereiteten Code ein.

Ihre Lösung:


In [ ]:
# Tragt die fachlich begründeten Eingabespalten ein.
merkmale = [] # Hier ihre Lösung
print("Gewählte Modellmerkmale:", merkmale)

### 3b) Bereinigten Baum und lineares Modell vergleichen

Trainiert zuerst denselben Baum nur mit den ausgewählten Merkmalen. Dadurch entsteht eine gültige Referenz für die Alternative. Die ursprüngliche Baseline mit allen Spalten bleibt als Beispiel für eine irreführende Bewertung erhalten.

Für die logistische Regression behandeln wir `cp` als Kategorie mit einer One-Hot-Codierung. Dies empfiehlt sich bei kategorialen Daten, da keine künstliche Ordnung und gleiche Abstände zwischen den Codes angenommen werden. Die übrigen Größen werden skaliert. Die `Pipeline` passt diese Schritte ausschließlich auf dem Training an.

Vergleicht mehrere Werte für `C`; ein kleineres `C` bedeutet stärkere Regularisierung - das genaue Verständnis ist nicht entscheidend. Wichtig ist nur, dass es sich hier um einen Hyperparameter handelt. Ergänzt die markierten Schritte.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score

kategorial = ["cp"]
numerisch = [name for name in merkmale if name not in kategorial]

vorverarbeitung = ColumnTransformer([
    ("skaliert", StandardScaler(), numerisch),
    ("kategorien", OneHotEncoder(handle_unknown="ignore"), kategorial),
])

bereinigte_baseline = Pipeline([
    ("vorverarbeitung", vorverarbeitung),
    ("klassifikator", DecisionTreeClassifier(max_depth=3, random_state=42)),
])
bereinigte_baseline.fit(X_train[merkmale], y_train)
vorhersage_bereinigt_val = bereinigte_baseline.predict(X_val[merkmale])
print("Bereinigter Baum, Balanced Accuracy:",
      round(balanced_accuracy_score(y_val, vorhersage_bereinigt_val), 3))
print("Bereinigter Baum, Recall Klasse 1:",
      round(recall_score(y_val, vorhersage_bereinigt_val), 3))
print("=======================================")
print("Ergebnisse der logistischen Regression:")

C_werte = [0] # Hier dein Code
modelle = []
ergebnisse = []
for C in C_werte:
    modell = Pipeline([
        ("vorverarbeitung", vorverarbeitung),
        ("klassifikator", LogisticRegression(C=C, class_weight="balanced", max_iter=1000)),
    ])
    # Ergänzt Modellanpassung und Vorhersage.
    modell.fit(0) # Hier dein Code
    vorhersage = modell.predict(0) # Hier dein Code
    modelle.append(modell)
    ergebnisse.append({
        "C": C,
        "Balanced Accuracy": balanced_accuracy_score(y_val, vorhersage),
        "Recall Klasse 1": recall_score(y_val, vorhersage),
    })

vergleich = pd.DataFrame(ergebnisse)
display(vergleich.round(3))


Wählt anhand der Validierung ein `C`. Berücksichtigt neben der balancierten Genauigkeit den Recall positiver Fälle. Wie schneidet die Alternative gegenüber dem bereinigten Baum ab? Tragt eure Wahl im folgenden Code ein und schaut euch die Confusion Matrix an.


Ihre Lösung:


In [ ]:
# Tragt hier den gewählten Wert aus C_werte ein.
gewaehltes_C = 0 # Hier dein Code
auswahl = C_werte.index(gewaehltes_C)
modell_gewaehlt = modelle[auswahl]
vorhersage_modell_val = modell_gewaehlt.predict(X_val[merkmale])
display(pd.DataFrame(
    confusion_matrix(y_val, vorhersage_modell_val, labels=[0, 1]),
    index=["wahr 0", "wahr 1"], columns=["Vorhersage 0", "Vorhersage 1"]
))


### 3c) Gruppenbezogene Fehler prüfen

Vergleicht für die beiden im Ausgangsdatensatz codierten Geschlechtsgruppen den Recall der positiven Klasse. Die gegebene Funktion `gruppen_recall` gibt auch die Zahl tatsächlich positiver Fälle aus, damit ihr kleine Nenner erkennt. `sex` dient hier als Prüfvariable und ist unabhängig davon verfügbar, ob ihr es als Modelleingabe gewählt habt.


In [ ]:
def gruppen_recall(y_wahr, y_prognose, gruppen):
    """Calculate recall for class 1 separately for groups 0 and 1.

    Parameters
    ----------
    y_wahr : pandas.Series
        True binary labels (0 or 1).
    y_prognose : numpy.ndarray
        Predicted binary labels, in the same row order as y_wahr.
    gruppen : pandas.Series
        Group membership (0 or 1), in the same row order as y_wahr.

    Returns
    -------
    pandas.DataFrame
        One row per group, with the group value ("sex"), the number of
        actual positive cases ("tatsächlich positiv"), and recall for
        class 1 ("Recall Klasse 1")."""
    zeilen = []
    for gruppe in [0, 1]:
        maske = (gruppen.to_numpy() == gruppe)
        positive = int(y_wahr.to_numpy()[maske].sum())
        recall = recall_score(y_wahr.to_numpy()[maske], y_prognose[maske], zero_division=0)
        zeilen.append({"sex": gruppe, "tatsächlich positiv": positive, "Recall Klasse 1": recall})
    return pd.DataFrame(zeilen)

print("Decision Tree, Validierung")
display(gruppen_recall(0)) # Hier dein Code
print("Logistische Regression, Validierung")
display(gruppen_recall(0)) # Hier dein Code


Unterscheiden sich die Gruppen-Recalls? Wie belastbar ist der Vergleich angesichts der Fallzahlen? Benennt mindestens eine zusätzliche Prüfung, die vor einem klinischen Einsatz nötig wäre.


Ihre Lösung:


### 3d) Abschließender Vergleich auf den Testdaten

Nach der Modellauswahl trainieren wir den bereinigten Baum und die gewählte logistische Regression erneut auf Training **und** Validierung. Erst danach betrachten wir die bisher zurückgehaltenen Testdaten.


In [ ]:
from sklearn.base import clone

abschluss_basis = DecisionTreeClassifier(max_depth=3, random_state=42)
abschluss_basis.fit(0) # Hier dein Code

abschluss_modell = clone(modell_gewaehlt)
abschluss_modell.fit(0) # Hier dein Code

prognose_basis_test = abschluss_basis.predict(0) # Hier dein Code
prognose_modell_test = abschluss_modell.predict(0) # Hier dein Code

for name, prognose in [("Decision Tree", prognose_basis_test), ("Logistische Regression", prognose_modell_test)]:
    print("==============================================")
    print(name)
    display(pd.DataFrame(confusion_matrix(y_test, prognose, labels=[0, 1]),
                         index=["wahr 0", "wahr 1"], columns=["Vorhersage 0", "Vorhersage 1"]))
    print("Balanced Accuracy:", round(balanced_accuracy_score(y_test, prognose), 3))
    print("Recall Klasse 1:", round(recall_score(y_test, prognose), 3))
    display(gruppen_recall(y_test, prognose, X_test["sex"]).round(3))


Vergleiche abschließend die beiden Modelle. Welche Aussagen kannst du treffen bzgl. der Gesamtperformance und der Performance auf dem geschützten Attribut? Kann man daraus finale Aussagen ziehen? Warum (nicht)?

Ihre Lösung:
